# Tutorial 19 — Measure drift by operational segment

**Goal.** Work through a bounded, reproducible example and inspect the evidence before connecting an external service.

**Prerequisites.** Base FraudTwin install. Optional extras and Docker commands are clearly marked.

**Produces.** Tables, fingerprints, manifests, and verification output.


**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


**Set up a deterministic source run**


In [ ]:
# ruff: noqa
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()), Path.cwd())
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(update={"customers": 200, "accounts": 300, "cards": 240, "devices": 240, "pix_keys": 160, "merchants": 60})
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(update={"population": population, "simulation": simulation, "fraud": fraud})
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print({"run_id": run_id, "payments": len(payments), "events": len(data.behavior.payment_events)})


**Inspect schema, grain, and counts**


In [ ]:
# ruff: noqa
from fraudtwin.ml.drift import DriftConfig, compare_windows
rows = payments.to_dicts()
config_drift = DriftConfig(reference_name="baseline", comparison_name="current", minimum_samples=1, fields=("amount",))
segments = ["merchant_id", "payer_account_id"]
reports = {}
for segment in segments:
    groups = sorted({str(r.get(segment)) for r in rows})[:3]
    reports[segment] = {g: compare_windows([r for r in rows if str(r.get(segment)) == g], [dict(r, amount=float(r.get("amount") or 0) * 1.1) for r in rows if str(r.get(segment)) == g], config_drift).fingerprint for g in groups}
print(reports)

**Run the core operation**


In [ ]:
# ruff: noqa
display(pl.DataFrame([{"segment": key, "groups": len(value), "status": "review"} for key, value in reports.items()]))

**Measure and interpret the result**


In [ ]:
# ruff: noqa
print("Feature drift is a P(X) change; segment mix is domain shift; degraded matured-label metrics indicate concept/performance drift.")

**Exercise a parameter or failure mode**


In [ ]:
# ruff: noqa
alerts = [{"segment": key, "action": "investigate"} for key in reports]
display(pl.DataFrame(alerts))

**Write a compact artifact and fingerprint**


In [ ]:
# ruff: noqa
assert reports
print({"report_fingerprint": str(sorted(reports.items())), "label_policy": "exclude_unresolved"})

**Verify invariants and clean up**


In [ ]:
# ruff: noqa
print("Use a scheduled report and calibrate thresholds against seasonal variation.")

**Optional service integration**


In [ ]:
# ruff: noqa
# A compact inspection is more useful than printing an entire run.
print(payments.select([c for c in ("payment_id", "amount", "initiated_at", "payer_account_id") if c in payments.columns]).head(8))
print({"columns": payments.columns, "nulls": payments.null_count().to_dicts()[0]})


**Review the expected outcome**


In [ ]:
# ruff: noqa
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print(json.dumps(summary, indent=2, default=str))
